In [1]:
import argparse
from pathlib import Path
from Utils.config_loader import load_config
from Backtest import SignalGenerator, BacktestEngine, PredictionMetrics, BacktestVisualizer


libgomp: Invalid value for environment variable OMP_NUM_THREADS

libgomp: Invalid value for environment variable OMP_NUM_THREADS


In [2]:
backtest_config = '/root/lio/Trade_LOB_MultiModal/Configs/backtest_config.yaml'
config = load_config(backtest_config)

In [3]:
# 创建输出目录
result_dir = Path(config.get('output', {}).get('result_dir', 'Backtest/result'))
log_dir = Path(config.get('output', {}).get('log_dir', 'Backtest/log'))
result_dir.mkdir(parents=True, exist_ok=True)
log_dir.mkdir(parents=True, exist_ok=True)

In [4]:
# Step 1: 生成信号
from Model.TCN_deeplobencoder import DeepLOB_TCN
from Model.TCN_lobencoder import LOB_TCN
# from torchinfo import summary
from Utils.config_loader import load_config
# from pathlib import Path
# print("加载配置...")
model_version = 'TCN_multi_modal_model_1'
model_config_path = f'/root/lio/Trade_LOB_MultiModal/checkpoints/{model_version}/model_config.yaml'
model_config = load_config(model_config_path)
model = LOB_TCN(model_config,num_classes = 3)
print("=" * 60)
print("Step 1: 生成预测信号")
print("=" * 60)
sig_gen = SignalGenerator(config)
signals, lob_data, labels_ret, time_bucket = sig_gen.run(model_version,model)

Step 1: 生成预测信号
模型加载完成: ./checkpoints/TCN_multi_modal_model_1/seed_42/best_model.pt
  验证指标: {'accuracy': 0.45828008655795804, 'precision_Down': np.float64(0.3829169087739686), 'recall_Down': np.float64(0.2783610909968622), 'f1_Down': np.float64(0.32237324854117894), 'support_Down': np.int64(16572), 'precision_Stationary': np.float64(0.48139040276485445), 'recall_Stationary': np.float64(0.8703785299419187), 'f1_Stationary': np.float64(0.6199164087128939), 'support_Stationary': np.int64(24965), 'precision_Up': np.float64(0.3683510638297872), 'recall_Up': np.float64(0.03229944029850746), 'f1_Up': np.float64(0.05939108061749571), 'support_Up': np.int64(17152), 'precision_macro': 0.4108861251228701, 'recall_macro': 0.3936796870790961, 'f1_macro': 0.3338935792905229, 'precision_weighted': 0.4205484479891514, 'recall_weighted': 0.45828008655795804, 'f1_weighted': 0.37208435027165393, 'precision_updown': np.float64(0.37563398630187794), 'recall_updown': np.float64(0.15533026564768482), 'f1_updo

In [17]:
signals['signal_lob_idx'].values[0]

np.int64(5999)

In [5]:
# sig_gen = SignalGenerator(config)
# import numpy as np
# lob_data, labels_ret, time_bucket = sig_gen.load_data()
# lob_data = np.asarray(lob_data)
# labels_ret = np.asarray(labels_ret)
# time_bucket = np.asarray(time_bucket)
# import pandas as pd
# signals = pd.read_csv('/root/lio/Trade_LOB_MultiModal/Backtest/result/signals.csv')

In [6]:
## 把 lob_data 的
# lob_data = lob_data.reshape(lob_data.shape[0], -1)

In [7]:
lob_data.shape

(13823700, 2, 20)

In [8]:
lob_data_ask_bid = lob_data[:,0,0:2]

In [9]:
lob_data_ask_bid[0,0]

np.float64(3058.5)

In [10]:
# # 假设你的数据变量名是 lob_data
# # 创建一个索引序列
# import numpy as np
# idx = np.arange(lob_data.shape[1])

# # 交换索引 1 和 2 的位置
# idx[1], idx[2] = idx[2], idx[1]

# # 应用新索引进行重排
# # 注意：这会创建一个视图（或副本），对于 1300万行数据，请确保内存充足
# lob_data_ask_bid = lob_data[:, idx]

In [11]:
# Step 2: 运行回测
print("\n" + "=" * 60)
print("Step 2: 运行回测引擎")
print("=" * 60)
engine = BacktestEngine(config)
result = engine.run(signals, lob_data_ask_bid, time_bucket)


Step 2: 运行回测引擎
回测完成:
  总交易次数: 646
  最终资金: 931,873.80
  总收益率: -6.8126%


In [12]:
# 保存交易日志
result.trades.to_csv(log_dir / 'trade_log.csv', index=False)
result.signals_with_match.to_csv(log_dir / 'signal_market_log.csv', index=False)
result.equity_curve.to_csv(log_dir / 'equity_curve.csv', index=False)
print(f"交易日志已保存至: {log_dir}")

交易日志已保存至: checkpoints/TCN_multi_modal_model_1/Backtest/log


In [18]:

# Step 3: 计算指标
print("\n" + "=" * 60)
print("Step 3: 计算绩效指标")
print("=" * 60)
metrics = PredictionMetrics(
    risk_free_rate=config.get('backtest', {}).get('risk_free_rate', 0.0)
)
report, benchmark = metrics.generate_report(
    result, signals, labels_ret, time_bucket, lob_data_ask_bid, config
)

# Step 4: 可视化
print("\n" + "=" * 60)
print("Step 4: 生成可视化")
print("=" * 60)
viz = BacktestVisualizer(output_dir=str(result_dir))
viz.generate_all(result, signals, benchmark, lob_data=lob_data_ask_bid, time_bucket=time_bucket)
def print_summary(report: dict):
    """打印回测摘要。"""
    sig = report.get('signal_metrics', {})
    strat = report.get('strategy_metrics', {})

    print("\n" + "=" * 60)
    print("回测结果摘要")
    print("=" * 60)

    print("\n--- 信号质量 ---")
    print(f"  信号总数:         {sig.get('signal_count_total', 0)}")
    print(f"  非平信号数:       {sig.get('signal_count_non_stationary', 0)}")
    print(f"  信号胜率:         {sig.get('signal_win_rate', 0):.4f}")
    print(f"  信号盈亏比:       {sig.get('signal_profit_loss_ratio', 0):.4f}")
    print(f"  预测准确率:       {sig.get('signal_accuracy', 0):.4f}")

    print("\n--- 实战收益 ---")
    print(f"  初始资金:         {strat.get('initial_capital', 0):,.2f}")
    print(f"  最终权益:         {strat.get('final_equity', 0):,.2f}")
    print(f"  累计收益率:       {strat.get('cumulative_return', 0) * 100:.4f}%")
    print(f"  年化收益率:       {strat.get('annualized_return', 0) * 100:.4f}%")
    print(f"  最大回撤:         {strat.get('max_drawdown', 0) * 100:.4f}%")
    print(f"  夏普比率:         {strat.get('sharpe_ratio', 0):.4f}")
    print(f"  交易次数:         {strat.get('trade_count', 0)}")
    print(f"  交易胜率:         {strat.get('trade_win_rate', 0):.4f}")
    print(f"  交易盈亏比:       {strat.get('trade_profit_loss_ratio', 0):.4f}")
    print(f"  日均收益率:       {strat.get('daily_return', 0) * 100:.6f}%")

    print(f"\n--- 基准对比 ---")
    print(f"  买入持有收益率:   {report.get('benchmark_return', 0) * 100:.4f}%")
    print(f"  超额收益:         {(strat.get('cumulative_return', 0) - report.get('benchmark_return', 0)) * 100:.4f}%")
    print("=" * 60)
# 打印摘要
print_summary(report)


Step 3: 计算绩效指标
buy_price:  3052.73
shares:  327.57564540591534
equity:  999998.3621217728
指标报告已保存至: checkpoints/TCN_multi_modal_model_1/Backtest/result/metrics_report.json

Step 4: 生成可视化
生成可视化图表:
  图表已保存: checkpoints/TCN_multi_modal_model_1/Backtest/result/equity_curve.png
  图表已保存: checkpoints/TCN_multi_modal_model_1/Backtest/result/drawdown.png
  图表已保存: checkpoints/TCN_multi_modal_model_1/Backtest/result/signal_distribution.png
  图表已保存: checkpoints/TCN_multi_modal_model_1/Backtest/result/pnl_distribution.png
  图表已保存: checkpoints/TCN_multi_modal_model_1/Backtest/result/cumulative_pnl.png
  图表已保存: checkpoints/TCN_multi_modal_model_1/Backtest/result/signal_midprice_detail.png
所有图表生成完成

回测结果摘要

--- 信号质量 ---
  信号总数:         7677
  非平信号数:       2023
  信号胜率:         0.2289
  信号盈亏比:       0.9723
  预测准确率:       0.6871

--- 实战收益 ---
  初始资金:         1,000,000.00
  最终权益:         931,875.36
  累计收益率:       -6.8125%
  年化收益率:       -80.0413%
  最大回撤:         16.3606%
  夏普比率:         -2.1090
  交易次数:  

In [14]:
signals['signal_lob_idx'].values,

(array([    5999,     7799,     9599, ..., 13819199, 13820999, 13822799],
       shape=(7677,)),)